# PhysX-Omni ????????? Notebook

?? notebook ?? `00_reviewer_soul_questions.md` ?????????????????????????????????

- benchmark ?? VLM judge ????????
- DQS/APS/MPS/KPS prompt ?????????
- ?? demo ???? URDF/XML/JSON ?????
- RLE ??????? 64? voxel ? token ???

?????https://arxiv.org/abs/2605.21572v1


In [1]:
from pathlib import Path
import json, re, xml.etree.ElementTree as ET
from collections import Counter, defaultdict

ROOT = Path.cwd()
if ROOT.name == "physx_omni_step9_reviewer":
    ROOT = ROOT.parent
REPO = ROOT / "physx-omni-assets" / "code" / "PhysX-Omni"
DEMO = Path(r"C:\Users\robot\physx_outputs\official_demo_full")
OUT = ROOT / "physx_omni_step9_reviewer"
REPORT_PATH = OUT / "reviewer_audit_report.json"

paths = {
    "repo": REPO,
    "benchmark_readme": REPO / "benchmark" / "README.md",
    "vlm_runner": REPO / "benchmark" / "code" / "vlm" / "multi.py",
    "aggregate": REPO / "benchmark" / "code" / "aggregation" / "aggregate_vlm_results.py",
    "prompts": REPO / "benchmark" / "prompts",
    "rle_script": REPO / "dataset" / "3generate_data_new_64_finetune_rle.py",
    "demo": DEMO,
    "basic_info": DEMO / "basic_info.json",
    "urdf": DEMO / "basic.urdf",
    "xml": DEMO / "basic.xml",
}

for name, path in paths.items():
    print(f"{name:18s} exists={path.exists()} path={path}")


repo               exists=True path=C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni
benchmark_readme   exists=True path=C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni\benchmark\README.md
vlm_runner         exists=True path=C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni\benchmark\code\vlm\multi.py
aggregate          exists=True path=C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni\benchmark\code\aggregation\aggregate_vlm_results.py
prompts            exists=True path=C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni\benchmark\prompts
rle_script         exists=True path=C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni\dataset\3generate_data_new_64_finetune_rle.py
demo               exists=True path=C:\Users\robot\physx_outputs\official_demo_full
basic_info         exists=True path=C:\Users\robot\physx_outputs\official_demo_full\basic_info.json
urdf               exists=True path=C:\Users\r

## 1. Benchmark ? judge ??

?????? benchmark ??????? VLM judge???????????? 0 ??

In [2]:
def read_text(path):
    return Path(path).read_text(encoding="utf-8", errors="replace")

readme = read_text(paths["benchmark_readme"])
vlm = read_text(paths["vlm_runner"])
agg = read_text(paths["aggregate"])

judge_hits = re.findall(r"Qwen/Qwen3\.5-122B-A10B", readme + "\n" + vlm)
missing_policy = [line.strip() for line in readme.splitlines() if "missing" in line.lower() and ("score 0" in line.lower() or "become" in line.lower())]
auto_score_terms = {term: vlm.count(term) for term in ["APS", "KPS", "MPS", "DCS", "RQS", "MCS", "AutoScore", "score=0"]}
agg_terms = {term: agg.count(term) for term in ["MPS", "DCS", "KPS", "APS", "DQS", "weighted_score", "25.0"]}

print("Default judge mentions:", len(judge_hits))
print("Missing evidence policy lines:")
for line in missing_policy[:12]:
    print("-", line)
print("\nAuto-score / metric term counts in multi.py:")
print(auto_score_terms)
print("\nAggregator term counts:")
print(agg_terms)


Default judge mentions: 9
Missing evidence policy lines:
- - missing RQS/MCS rendered views become score 0;
- - missing DCS color/mask/description evidence becomes DCS 0;
- - malformed or missing DQS generated dimensions become DQS 0;
- - missing APS heatmaps become APS 0;
- - missing KPS videos become KPS 0 for active methods;
- - missing MPS water/floor video pairs become MPS 0.

Auto-score / metric term counts in multi.py:
{'APS': 10, 'KPS': 14, 'MPS': 9, 'DCS': 9, 'RQS': 5, 'MCS': 4, 'AutoScore': 5, 'score=0': 5}

Aggregator term counts:
{'MPS': 8, 'DCS': 7, 'KPS': 3, 'APS': 5, 'DQS': 3, 'weighted_score': 6, '25.0': 4}


## 2. Prompt ????????

????? DQS/APS/MPS/KPS prompt ??????????? prompt ??????????? benchmark ???????/???????????????????????

In [3]:
prompt_dir = paths["prompts"]
keywords = [
    "category priors", "common everyday dimensions", "common sense",
    "material physics", "common material", "uncertainty",
    "image-based material priors", "likely_motion", "observed_state",
    "Young", "Poisson", "density", "VAPS", "APS", "DQS", "MPS",
]

prompt_hits = defaultdict(list)
for p in sorted(prompt_dir.glob("*.yaml")):
    text = read_text(p)
    for i, line in enumerate(text.splitlines(), 1):
        low = line.lower()
        if any(k.lower() in low for k in keywords):
            prompt_hits[p.name].append((i, line.strip()))

for name, hits in prompt_hits.items():
    print(f"\n{name}: {len(hits)} hits")
    for line_no, line in hits[:10]:
        print(f"  L{line_no}: {line[:180]}")



prompts_affordance.yaml: 20 hits
  L6: [Prompt: image + affordance heatmap -> final APS]
  L21: APS = Affordance Plausibility Score, ranging from 0 to 100.
  L44: - If a region may cause harm to the human body according to common sense, it should usually be regarded as having lower affordance.
  L74: Based only on the original image and everyday common sense, judge:
  L89: Combine the reasonable parts and the unreasonable parts, and give a balanced final APS score.
  L93: Core meaning of APS:
  L94: - High score: the relative ranking in the heatmap is highly consistent with everyday common sense, the priority of key regions is reasonable, and dangerous regions are not highligh
  L98: APS score interpretation:
  L99: - 90–100: the relative ranking is highly reasonable, and the overall result strongly matches common sense
  L103: - 0–39: very poor, with an overall ranking that clearly violates common sense, or dangerous / unreasonable regions being severely highlighted incorrectly

prom

## 3. ?? demo JSON/URDF/XML ??????

??????????? demo ?????? `basic_info.json` ???????????? `basic.urdf` / `basic.xml` ???????

In [4]:
def parse_urdf(path):
    root = ET.parse(path).getroot()
    links = root.findall("link")
    joints = root.findall("joint")
    masses, inertias = [], []
    friction_attrs = []
    limits = []
    for elem in root.iter():
        for k, v in elem.attrib.items():
            if "friction" in k.lower():
                friction_attrs.append((elem.tag, k, v))
    for link in links:
        inertial = link.find("inertial")
        if inertial is not None:
            m = inertial.find("mass")
            if m is not None and "value" in m.attrib:
                masses.append(float(m.attrib["value"]))
            iner = inertial.find("inertia")
            if iner is not None:
                inertias.append(tuple((k, iner.attrib.get(k)) for k in sorted(iner.attrib)))
    for j in joints:
        lim = j.find("limit")
        if lim is not None:
            limits.append(dict(lim.attrib))
    return {
        "links": len(links),
        "joints": len(joints),
        "joint_types": dict(Counter(j.attrib.get("type", "") for j in joints)),
        "masses": masses,
        "unique_masses": sorted(set(masses)),
        "inertia_count": len(inertias),
        "unique_inertia_count": len(set(inertias)),
        "limits": limits,
        "friction_attrs": friction_attrs,
    }

def parse_mjcf(path):
    root = ET.parse(path).getroot()
    geoms = root.findall(".//geom")
    joints = root.findall(".//joint")
    density = [g.attrib.get("density") for g in geoms if "density" in g.attrib]
    mass = [g.attrib.get("mass") for g in geoms if "mass" in g.attrib]
    inertia = [b.attrib.get("inertia") for b in root.findall(".//body") if "inertia" in b.attrib]
    friction_attrs = []
    for elem in root.iter():
        for k, v in elem.attrib.items():
            if "friction" in k.lower():
                friction_attrs.append((elem.tag, k, v))
    return {
        "geoms": len(geoms),
        "joints": len(joints),
        "joint_types": dict(Counter(j.attrib.get("type", "") for j in joints)),
        "densities": density,
        "geom_mass_attrs": mass,
        "body_inertia_attrs": inertia,
        "friction_attrs": friction_attrs,
    }

urdf_audit = parse_urdf(paths["urdf"])
xml_audit = parse_mjcf(paths["xml"])

print("URDF audit:")
print(json.dumps(urdf_audit, indent=2, ensure_ascii=False)[:3000])
print("\nMJCF/XML audit:")
print(json.dumps(xml_audit, indent=2, ensure_ascii=False)[:3000])


URDF audit:
{
  "links": 13,
  "joints": 12,
  "joint_types": {
    "fixed": 7,
    "continuous": 4,
    "revolute": 1
  },
  "masses": [
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0
  ],
  "unique_masses": [
    1.0
  ],
  "inertia_count": 13,
  "unique_inertia_count": 1,
  "limits": [
    {
      "effort": "2000.0",
      "velocity": "2.0"
    },
    {
      "lower": "-3.141592653589793",
      "upper": "0.0",
      "effort": "2000.0",
      "velocity": "2.0"
    },
    {
      "effort": "2000.0",
      "velocity": "2.0"
    },
    {
      "effort": "2000.0",
      "velocity": "2.0"
    },
    {
      "effort": "2000.0",
      "velocity": "2.0"
    }
  ],
  "friction_attrs": []
}

MJCF/XML audit:
{
  "geoms": 16,
  "joints": 5,
  "joint_types": {
    "hinge": 5
  },
  "densities": [
    "4450.0",
    "7800.0",
    "950.0",
    "4450.0",
    "7800.0",
    "4450.0",
    "4450.0"
  ],
  "geom_mass_attrs": [],
  "body

In [5]:
basic_info = json.loads(read_text(paths["basic_info"]))
parts = basic_info.get("parts", [])
print("object:", basic_info.get("object"))
print("part_count:", len(parts))
for idx, part in enumerate(parts):
    if idx >= 12:
        break
    print(idx, {
        "name": part.get("name"),
        "material": part.get("material"),
        "density": part.get("density"),
        "youngs_modulus": part.get("youngs_modulus"),
        "poisson_ratio": part.get("poisson_ratio"),
    })
print("group_info_type:", type(basic_info.get("group_info")).__name__)
print("group_info_count:", len(basic_info.get("group_info", {})) if isinstance(basic_info.get("group_info"), dict) else "n/a")


object: None
part_count: 7
0 {'name': 'Caster Wheel (Front Right)', 'material': 'Steel with Rubber', 'density': 4.45, 'youngs_modulus': None, 'poisson_ratio': None}
1 {'name': 'Main Body', 'material': 'Steel', 'density': 7.8, 'youngs_modulus': None, 'poisson_ratio': None}
2 {'name': 'Lid', 'material': 'Plastic (HDPE)', 'density': 0.95, 'youngs_modulus': None, 'poisson_ratio': None}
3 {'name': 'Caster Wheel (Front Left)', 'material': 'Steel with Rubber', 'density': 4.45, 'youngs_modulus': None, 'poisson_ratio': None}
4 {'name': 'Front Bar Handle', 'material': 'Steel', 'density': 7.8, 'youngs_modulus': None, 'poisson_ratio': None}
5 {'name': 'right rear wheel', 'material': 'Steel with Rubber', 'density': 4.45, 'youngs_modulus': None, 'poisson_ratio': None}
6 {'name': 'Caster Wheel (Rear Left)', 'material': 'Steel with Rubber', 'density': 4.45, 'youngs_modulus': None, 'poisson_ratio': None}
group_info_type: dict
group_info_count: 6


## 4. RLE ????

????????????? 64? voxel????? template/delta lossless ??????? token ???

In [6]:
rle = read_text(paths["rle_script"])
patterns = {
    "shape_64": r"shape:\s*Tuple\[int, int, int\]\s*=\s*\(64, 64, 64\)",
    "encode_func": r"def encode_voxel_2drle_by_z",
    "decode_func": r"def decode_voxel_2drle_by_z",
    "lossless_comment": r"Lossless template encoding",
    "assert_equal": r"assert_runs_by_z_equal|np\.array_equal",
    "token_filter_20000": r"tokennum\s*<\s*20000",
    "grid_64_prompt": r"grid=64",
}

rle_audit = {}
for name, pat in patterns.items():
    hits = [(i, line.strip()) for i, line in enumerate(rle.splitlines(), 1) if re.search(pat, line)]
    rle_audit[name] = hits[:8]
    print(f"{name}: {len(hits)} hits")
    for line_no, line in hits[:4]:
        print(f"  L{line_no}: {line}")


shape_64: 2 hits
  L261: shape: Tuple[int, int, int] = (64, 64, 64),  # (D,H,W) = (z,y,x)
  L330: shape: Tuple[int, int, int] = (64, 64, 64),
encode_func: 1 hits
  L259: def encode_voxel_2drle_by_z(
decode_func: 1 hits
  L328: def decode_voxel_2drle_by_z(
lossless_comment: 1 hits
  L91: # Lossless template encoding with delta
assert_equal: 2 hits
  L245: def assert_runs_by_z_equal(a: List[np.ndarray], b: List[np.ndarray]):
  L507: logger.info(name+'_'+str(part)+'_'+str(ind)+'_same: '+str(np.array_equal(np.unique(idx), np.unique(idx2))))
token_filter_20000: 1 hits
  L520: if tokennum<20000:
grid_64_prompt: 2 hits
  L512: "Based on the structured description of l_"+str(part)+", generate its 3D voxel (grid=64) in the 3D RLE (linear scan) format. Output one run per line as: start_index length",\
  L516: tokennum=len(tokenizer(str(txt)+"Based on the structured description of l_"+str(part)+", generate its 3D voxel (grid=64) in the 3D RLE (linear scan) format. Output one run per line as: star

## 5. ?????????????

In [7]:
questions = [
    ("Q1", "single-image physical properties", "mostly plausible priors, not calibrated measurements", "dimension/material/affordance prompts explicitly rely on priors and common knowledge"),
    ("Q2", "VLM judge stability", "not proven", "default Qwen3.5 judge is clear; multi-judge rank correlation is needed"),
    ("Q3", "cross-simulator consistency", "not proven", "benchmark uses simulator evidence, but not equivalent to MuJoCo/Isaac/Genesis consistency"),
    ("Q4", "URDF/XML physical completeness", "insufficient in local demo", "URDF mass all 1.0 and repeated inertia; XML density exists but frictionloss is 0.0"),
    ("Q5", "real robot sim-to-real", "not proven by local evidence", "needs generated vs scanned/handcrafted vs baseline robot ablation"),
    ("Q6", "template RLE generalization", "encoding is solid; scaling unproven", "64^3 and token<20000 indicate resolution/token bottleneck"),
    ("Q7", "stronger decoder bottleneck", "bottleneck shifts", "geometry improves, but physical calibration/joints/simulator files/judge remain bottlenecks"),
]

for qid, topic, verdict, evidence in questions:
    print(f"{qid:>2} | {topic:35s} | {verdict}")
    print(f"   evidence: {evidence}")


Q1 | single-image physical properties    | mostly plausible priors, not calibrated measurements
   evidence: dimension/material/affordance prompts explicitly rely on priors and common knowledge
Q2 | VLM judge stability                 | not proven
   evidence: default Qwen3.5 judge is clear; multi-judge rank correlation is needed
Q3 | cross-simulator consistency         | not proven
   evidence: benchmark uses simulator evidence, but not equivalent to MuJoCo/Isaac/Genesis consistency
Q4 | URDF/XML physical completeness      | insufficient in local demo
   evidence: URDF mass all 1.0 and repeated inertia; XML density exists but frictionloss is 0.0
Q5 | real robot sim-to-real              | not proven by local evidence
   evidence: needs generated vs scanned/handcrafted vs baseline robot ablation
Q6 | template RLE generalization         | encoding is solid; scaling unproven
   evidence: 64^3 and token<20000 indicate resolution/token bottleneck
Q7 | stronger decoder bottleneck         | b

## 6. ?????????

??? notebook ????????? `reviewer_audit_report.json`??????????

In [8]:
report = {
    "paper": "https://arxiv.org/abs/2605.21572v1",
    "code": "https://github.com/physx-omni/PhysX-Omni",
    "local_repo": str(REPO),
    "local_demo": str(DEMO),
    "default_judge_mentions": len(judge_hits),
    "missing_policy_lines": missing_policy,
    "auto_score_terms": auto_score_terms,
    "aggregator_terms": agg_terms,
    "prompt_hit_counts": {k: len(v) for k, v in prompt_hits.items()},
    "urdf_audit": urdf_audit,
    "xml_audit": xml_audit,
    "rle_audit_hit_counts": {k: len(v) for k, v in rle_audit.items()},
    "reviewer_questions": [
        {"id": qid, "topic": topic, "verdict": verdict, "evidence": evidence}
        for qid, topic, verdict, evidence in questions
    ],
}
REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("wrote", REPORT_PATH)
print(json.dumps({"default_judge_mentions": len(judge_hits), "prompt_files": len(prompt_hits), "urdf_unique_masses": urdf_audit["unique_masses"]}, ensure_ascii=False, indent=2))


wrote C:\Users\robot\Documents\成长学习库\physx_omni_step9_reviewer\reviewer_audit_report.json
{
  "default_judge_mentions": 9,
  "prompt_files": 6,
  "urdf_unique_masses": [
    1.0
  ]
}
